# S6E9 Origin Is Synthetic Too

The Data page for this Playground competition says the train and test files were
generated by a model trained on a public dataset — **EV Adoption Behavior and Range
Anxiety** — and that the feature distributions are close to, but not the same as, that
source. The obvious next move is to go read the source. So I did, and the first thing
worth saying about it is that it is not real either.

Its own Kaggle page says so in its first line: "This dataset provides a comprehensive,
synthetic snapshot of 10,000 potential car buyers"
(`itzzomkar/ev-adoption-behavior-and-range-anxiety`). [Georgy Mamarin's S6E9](https://www.kaggle.com/code/georgymamarin/s6e9-starter-how-to-tell-a-real-gain-from-noise) starter
notebook makes the same point.

So the chain is:

```
a synthetic file of 10,000 rows  ->  Kaggle's generator  ->  train.csv
```

There is no real-world layer under any of it. That changes what these 10,000 rows are
good for. They are not ground truth and they are not a ceiling; they are the **first
draw** from a process this competition takes a second draw from, and the useful question
is what structure survives that second draw.

What is in here:

- the 10,000 source rows, profiled column by column, including what each categorical
  level does to the buy rate
- a fixed-seed 5-fold measurement of how much signal is in the source file — a
  logistic regression against a four-step LightGBM capacity ladder
- a controlled demonstration of the one mistake that makes "append the source data" look
  like it works when it does not
- `original_clean.csv` written to `/kaggle/working`, so you can mount this notebook's
  output instead of re-doing the cleaning

The number in the title is the best out-of-fold AUC computed in the baseline section
below. The notebook asserts its own headline at run time, so if the claim ever stops
being true the cell fails instead of quietly disagreeing with the title.

⚠️ The quoted sentence above is my transcription of the dataset page; read it yourself
before quoting me. It is the only claim about the source's provenance I am making, and
everything below rests on it.

**This notebook does not touch the competition files.** Nothing here tells you what the
source data is worth *on S6E9* — that needs `train.csv`, and it is the next notebook, not
this one. I would rather leave that blank than guess it.


## The one cell you edit

Everything below reads `CFG`. The knob that costs real time is `CONTAM_CONFIGS` — its
last rung is an over-powered model and it is most of the runtime. Drop it if you are in
a hurry; the notebook still runs and prints one fewer row.


In [1]:
CFG = {
    "SEED": 42,          # fold seed. The whole notebook is deterministic at this seed.
    "N_SPLITS": 5,       # StratifiedKFold splits for every measurement here.
    "TARGET": "Will_Buy_EV",
    "ID_COL": "Buyer_ID",   # the original's id column. The competition's is "id".

    # The capacity ladder: num_leaves for each rung, coarsest last.
    # These are fixed, not tuned. Nothing in this notebook searches over them,
    # because a search inside a notebook whose prose quotes the result is how you
    # end up publishing a number your own code no longer produces.
    "LADDER": [31, 8, 4, 2],

    # The "does appending source rows help" experiment: what fraction of the source
    # file is held back to play the part of the extra data.
    "EXTRA_FRAC": 0.30,

    # Rows contaminated on purpose in the de-duplication experiment.
    "N_INJECT": 500,

    # The rungs the contamination experiment is run at. The third is deliberately
    # over-powered - it is there to answer "can this leak be reached at all", not
    # because anyone should fit it. Cutting this list is the cheapest speed-up.
    "CONTAM_CONFIGS": [
        ("stumps", dict(n_estimators=800, learning_rate=0.02, num_leaves=2,
                        min_child_samples=100, reg_lambda=10.0)),
        ("mid", dict(n_estimators=800, learning_rate=0.02, num_leaves=31,
                     min_child_samples=100, reg_lambda=10.0)),
        ("memoriser", dict(n_estimators=300, learning_rate=0.1, num_leaves=255,
                           min_child_samples=2, reg_lambda=0.0)),
    ],
}

for k, v in CFG.items():
    print(f"{k:>12} = {v!r}")


## Finding the file

Kaggle has more than one mount layout — `/kaggle/input/<slug>/` on older sessions,
`/kaggle/input/datasets/<owner>/<slug>/` on newer ones — so hard-coding a path is a
`FileNotFoundError` waiting for a session upgrade. This does a bounded breadth-first
search for the filename and prints what it found. If it finds nothing it prints the
mount tree and stops, rather than continuing with an empty frame.


In [2]:
import os, time, warnings
from collections import deque

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import lightgbm as lgb

warnings.filterwarnings("ignore")
T0 = time.time()

# Local dry runs set SMOKE_ROOT / SMOKE_WORK to directories on my laptop that mimic
# the Kaggle layout. On Kaggle neither is set and both fall back to the real paths.
SEARCH_ROOT = os.environ.get("SMOKE_ROOT", "/kaggle/input")
WORK = os.environ.get("SMOKE_WORK", "/kaggle/working")
os.makedirs(WORK, exist_ok=True)


def find_file(filename, root=SEARCH_ROOT, max_depth=5):
    """Bounded breadth-first search for `filename` under `root`."""
    queue = deque([(root, 0)])
    hits = []
    while queue:
        directory, depth = queue.popleft()
        try:
            entries = sorted(os.scandir(directory), key=lambda e: e.name)
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue
        for entry in entries:
            try:
                if entry.is_file() and entry.name == filename:
                    hits.append(entry.path)
                elif entry.is_dir() and depth < max_depth:
                    queue.append((entry.path, depth + 1))
            except OSError:
                continue
    if not hits:
        tree = []
        stack = deque([(root, 0)])
        while stack and len(tree) < 60:
            directory, depth = stack.popleft()
            try:
                entries = sorted(os.scandir(directory), key=lambda e: e.name)
            except OSError:
                continue
            for entry in entries:
                tree.append(entry.path)
                if entry.is_dir() and depth < 3:
                    stack.append((entry.path, depth + 1))
        raise FileNotFoundError(
            f"{filename!r} not found under {root}. What is mounted:\n  "
            + "\n  ".join(tree[:60]))
    hits.sort(key=len)
    return hits[0]


ORIGINAL_CSV = find_file("EV_Adoption_and_Range_Anxiety_Dataset.csv")
print("original:", ORIGINAL_CSV)

raw = pd.read_csv(ORIGINAL_CSV)
print(f"shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"memory: {raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

y = (raw[CFG["TARGET"]].astype(str).str.strip() == "Yes").astype(np.int8).to_numpy()
print(f"\npositive rate: {y.mean():.4%}  ({int(y.sum()):,} of {len(y):,} rows say Yes)")
print(f"target values seen: {sorted(raw[CFG['TARGET']].astype(str).unique())}")
print(f"duplicate {CFG['ID_COL']} values: {int(raw[CFG['ID_COL']].duplicated().sum())}")
print(f"[{time.time() - T0:.0f}s]")


## The 15 columns

One row per column: dtype, distinct values, how much is missing, and — for the numeric
columns — the AUC you get from ranking on that column alone. A single-column AUC below
0.5 means the column ranks the wrong way round, which is information, not a bug.

The competition files are wider than this, so treat this as the *core* schema rather than
the full one — anything in `train.csv` that is not below came from the generator, not from
here.


In [3]:
feature_cols = [c for c in raw.columns if c not in (CFG["TARGET"], CFG["ID_COL"])]
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(raw[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

rows = []
for c in raw.columns:
    s = raw[c]
    solo = ""
    if c in num_cols:
        v = pd.to_numeric(s, errors="coerce")
        v = v.fillna(v.median())
        solo = f"{roc_auc_score(y, v.to_numpy()):.4f}"
    rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "role": ("target" if c == CFG["TARGET"] else
                 "id" if c == CFG["ID_COL"] else
                 "numeric" if c in num_cols else "categorical"),
        "distinct": int(s.nunique(dropna=True)),
        "missing": int(s.isna().sum()),
        "missing_%": round(100.0 * s.isna().mean(), 2),
        "solo_AUC": solo,
    })

schema = pd.DataFrame(rows)
print(schema.to_string(index=False))
print(f"\n{len(num_cols)} numeric, {len(cat_cols)} categorical, "
      f"1 target, 1 id. Missing values in "
      f"{int((schema['missing'] > 0).sum())} columns.")

solo = schema.loc[schema["solo_AUC"] != "", ["column", "solo_AUC"]].copy()
solo["gap"] = (solo["solo_AUC"].astype(float) - 0.5).abs()
flat = solo.loc[solo["gap"] < 0.01, "column"].tolist()
print(f"\nnumeric columns whose solo AUC sits within 0.01 of a coin flip: "
      f"{len(flat)} of {len(solo)}")
if flat:
    print("  " + ", ".join(flat))


### What each categorical level does to the buy rate

The base rate is printed above. A level whose buy rate sits far from it is carrying
signal; a level that sits on top of it is not. `Range_Anxiety_Level` and
`Home_Charging_Possible` are the two to look at first — the whole premise of the dataset
is that range anxiety suppresses purchase intent, and the table says whether it does.


In [4]:
base = y.mean()
blocks = []
for c in cat_cols:
    g = pd.DataFrame({"level": raw[c].astype(str), "y": y})
    t = (g.groupby("level")["y"]
           .agg(n="size", buy_rate="mean")
           .reset_index()
           .sort_values("buy_rate", ascending=False))
    t.insert(0, "column", c)
    t["lift_vs_base"] = t["buy_rate"] - base
    blocks.append(t)

levels = pd.concat(blocks, ignore_index=True)
levels["buy_rate"] = levels["buy_rate"].round(4)
levels["lift_vs_base"] = levels["lift_vs_base"].round(4)
print(f"base buy rate = {base:.4f}\n")
print(levels.to_string(index=False))

spread = (levels.groupby("column")["buy_rate"]
                .agg(lambda s: s.max() - s.min())
                .sort_values(ascending=False))
print("\nbuy-rate spread within each categorical, widest first:")
print(spread.round(4).to_string())


## How much signal is in the source file?

Fixed protocol, no tuning anywhere in this notebook:

- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` over the file in its
  original row order
- one logistic regression: median-impute and standardise the numerics, most-frequent
  impute and one-hot the categoricals
- four LightGBM models that differ **only** in capacity — `num_leaves` 31, 8, 4, 2 —
  with everything else held fixed
- no early stopping on the validation fold, so no fold ever sees its own rows through a
  stopping rule

Out-of-fold AUC, plus the standard deviation across the five folds, so you can see
whether a difference between two rungs is bigger than the fold-to-fold noise.


In [5]:
skf = StratifiedKFold(n_splits=CFG["N_SPLITS"], shuffle=True,
                      random_state=CFG["SEED"])

X = raw[feature_cols].copy()
X_lgb = X.copy()
for c in cat_cols:
    X_lgb[c] = pd.Categorical(X_lgb[c].astype(object))
for c in num_cols:
    X_lgb[c] = pd.to_numeric(X_lgb[c], errors="coerce").astype("float64")

X_lin = X.copy()
for c in cat_cols:
    X_lin[c] = X_lin[c].astype(object)
for c in num_cols:
    X_lin[c] = pd.to_numeric(X_lin[c], errors="coerce")

splits = list(skf.split(X, y))
fold_id = np.zeros(len(y), dtype=int)
for f, (_, va) in enumerate(splits):
    fold_id[va] = f


def score(oof):
    per_fold = [roc_auc_score(y[fold_id == f], oof[fold_id == f])
                for f in range(CFG["N_SPLITS"])]
    return roc_auc_score(y, oof), float(np.std(per_fold, ddof=1)), per_fold


def run_lgb(num_leaves):
    oof = np.zeros(len(y))
    for tr, va in splits:
        model = lgb.LGBMClassifier(
            n_estimators=800, learning_rate=0.02, num_leaves=num_leaves,
            min_child_samples=100, reg_lambda=10.0, colsample_bytree=0.7,
            subsample=0.8, subsample_freq=1,
            random_state=CFG["SEED"], verbose=-1)
        model.fit(X_lgb.iloc[tr], y[tr])
        oof[va] = model.predict_proba(X_lgb.iloc[va])[:, 1]
    return oof


def run_logreg():
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])
    oof = np.zeros(len(y))
    for tr, va in splits:
        pipe = Pipeline([("pre", pre),
                         ("clf", LogisticRegression(max_iter=2000, C=1.0))])
        pipe.fit(X_lin.iloc[tr], y[tr])
        oof[va] = pipe.predict_proba(X_lin.iloc[va])[:, 1]
    return oof


results = []
oof_store = {}

oof_store["logreg"] = run_logreg()
auc, sd, per_fold = score(oof_store["logreg"])
results.append({"model": "logistic regression", "num_leaves": "-",
                "oof_auc": auc, "fold_sd": sd})
print(f"logistic regression      OOF {auc:.6f}  fold sd {sd:.6f}  "
      f"folds {[round(a, 5) for a in per_fold]}   [{time.time() - T0:.0f}s]")

for leaves in CFG["LADDER"]:
    key = f"lgb_{leaves}"
    oof_store[key] = run_lgb(leaves)
    auc, sd, per_fold = score(oof_store[key])
    results.append({"model": f"LightGBM", "num_leaves": leaves,
                    "oof_auc": auc, "fold_sd": sd})
    print(f"LightGBM num_leaves={leaves:<3}  OOF {auc:.6f}  fold sd {sd:.6f}  "
          f"folds {[round(a, 5) for a in per_fold]}   [{time.time() - T0:.0f}s]")

table = pd.DataFrame(results).sort_values("oof_auc", ascending=False)
BEST_ROW = table.iloc[0]
BEST_AUC = float(BEST_ROW["oof_auc"])
BEST_NAME = BEST_ROW["model"] + ("" if BEST_ROW["num_leaves"] == "-"
                                 else f" num_leaves={BEST_ROW['num_leaves']}")
FOLD_SD_MEAN = float(table["fold_sd"].mean())
print("\n" + table.to_string(index=False,
                             formatters={"oof_auc": "{:.6f}".format,
                                         "fold_sd": "{:.6f}".format}))
print(f"\nbest: {BEST_NAME} at {BEST_AUC:.6f}")
print(f"mean fold SD across the five models: {FOLD_SD_MEAN:.6f}")


In [6]:
# The headline check. The title of this notebook carries a number, and Kaggle will not
# let me rename a notebook without moving its slug, so the number has to keep being
# true. If this assert ever fires, the honest fix is a new notebook.
HEADLINE = f"{BEST_AUC:.3f}"
print("TITLE_AUC=" + HEADLINE)
assert HEADLINE == "0.905", (
    f"title says 0.905, this run computed {HEADLINE}")
print("headline agrees with the title.")


### The ladder, plotted

Error bars are one fold standard deviation. The point of the picture is the direction:
if the bars go *down* as capacity goes up, extra capacity is being spent on noise.


In [7]:
fig, ax = plt.subplots(figsize=(7.2, 3.6))
labels = [f"{r['model'].split()[0]}\n{r['num_leaves']}" for _, r in
          pd.DataFrame(results).iterrows()]
vals = [r["oof_auc"] for r in results]
errs = [r["fold_sd"] for r in results]
colors = ["#4c6ef5"] + ["#868e96"] * (len(results) - 1)
ax.bar(range(len(vals)), vals, yerr=errs, capsize=4, color=colors)
ax.set_xticks(range(len(vals)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(min(vals) - 0.02, max(vals) + 0.01)
ax.set_ylabel("out-of-fold AUC")
ax.set_title(f"Original dataset, {CFG['N_SPLITS']}-fold, seed {CFG['SEED']} "
             f"(bars: 1 fold SD)", fontsize=10)
ax.grid(axis="y", alpha=0.25)
for i, v in enumerate(vals):
    ax.text(i, v + errs[i] + 0.001, f"{v:.4f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

print(f"spread across the ladder: {max(vals) - min(vals):.6f} AUC")
print(f"mean fold SD:             {FOLD_SD_MEAN:.6f}")
print(f"the spread is {(max(vals) - min(vals)) / FOLD_SD_MEAN:.2f} fold SDs wide")


### Reading that

Three things in the table are worth carrying into the competition:

1. **Whether the linear model loses.** With 10,000 rows and a 17.5% base rate there are
   fewer than 2,000 positive cases in the whole file. Compare the top of the ladder
   against the bottom before assuming a big tree model is the right default here.
2. **Compare gaps to the fold SD.** The printed standard deviation is the scale on which
   differences in this file mean anything. Smaller than that is a fold accident.
3. **The competition file is much larger than this one.** Whatever capacity wins on
   10,000 rows is not automatically what wins on the competition's draw — more rows buy
   back capacity. Re-run the ladder on `train.csv` before you trust it there.

And one thing not to carry: this AUC is not an upper bound on S6E9. It is what a fixed
recipe gets on one 10,000-row synthetic file. Kaggle's generator fits its own model
to that draw and samples a much larger file, which can sharpen the structure as easily as
blur it. The competition's achievable AUC could land on either side of this number.


## The way Playground competitors use a source dataset

The standard recipe, and it is a recipe rather than a rule, is:

- append the source rows to the **training part of each fold only**
- never to the validation part, and never before the split
- de-duplicate the source against the competition rows first, because the generator can
  reproduce source rows and a row that lands in both halves of a fold is a leak

Note what the recipe is *not* doing here. Usually "add the original data" means adding
real observations to a synthetic set. On S6E9 both sides are synthetic, so appending this
file is adding a second synthetic draw — more rows from the same source, not a correction
towards reality. Whether that helps is an empirical question, not a free lunch.

I cannot measure what it is worth on S6E9 from here; that needs `train.csv`. What I can do
is run the same machinery inside this file: hold back 30% of it to play the part of the
extra rows, train on the other 70%, and see whether appending helps. And then show what
the number looks like when the de-duplication step is skipped.


In [8]:
rng = np.random.default_rng(CFG["SEED"])
perm = rng.permutation(len(raw))
n_extra = int(round(CFG["EXTRA_FRAC"] * len(raw)))
extra_idx = perm[:n_extra]
main_idx = np.sort(perm[n_extra:])

X_main = X_lgb.iloc[main_idx].reset_index(drop=True)
y_main = y[main_idx]
X_extra = X_lgb.iloc[extra_idx].reset_index(drop=True)
y_extra = y[extra_idx]

# The contaminated variant. Instead of *adding* copies - which would change the row
# count and make the two runs incomparable - it OVERWRITES N_INJECT positions of the
# main set with rows from the extra set. Same size, same folds, same everything,
# except that some main rows now have exact twins in the block that gets appended.
pos = np.random.default_rng(CFG["SEED"] + 1).choice(
    len(X_main), CFG["N_INJECT"], replace=False)
X_dirty = X_main.copy()
y_dirty = y_main.copy()
X_dirty.iloc[pos] = X_extra.iloc[:CFG["N_INJECT"]].to_numpy()
y_dirty[pos] = y_extra[:CFG["N_INJECT"]]

print(f"main set   {len(X_main):,} rows, positive rate {y_main.mean():.4f}")
print(f"extra set  {len(X_extra):,} rows, positive rate {y_extra.mean():.4f}")
print(f"dirty set  {len(X_dirty):,} rows, positive rate {y_dirty.mean():.4f}  "
      f"({CFG['N_INJECT']:,} of them duplicated from the extra set, "
      f"{CFG['N_INJECT'] / len(X_dirty):.2%})")


In [9]:
def fold_experiment(main_X, main_y, extra_X=None, extra_y=None, **params):
    """5-fold over main_X. Extra rows go into the TRAINING part of each fold only.

    Validation is always the untouched held-out slice of main_X, so the number this
    returns is comparable across every arm below.
    """
    inner = StratifiedKFold(n_splits=CFG["N_SPLITS"], shuffle=True,
                            random_state=CFG["SEED"])
    oof = np.zeros(len(main_y))
    for tr, va in inner.split(main_X, main_y):
        Xtr, ytr = main_X.iloc[tr], main_y[tr]
        if extra_X is not None:
            Xtr = pd.concat([Xtr, extra_X], ignore_index=True)
            ytr = np.concatenate([ytr, extra_y])
        model = lgb.LGBMClassifier(random_state=CFG["SEED"], verbose=-1, **params)
        model.fit(Xtr, ytr)
        oof[va] = model.predict_proba(main_X.iloc[va])[:, 1]
    return roc_auc_score(main_y, oof)


contam = []
for name, params in CFG["CONTAM_CONFIGS"]:
    a = fold_experiment(X_main, y_main, **params)
    b = fold_experiment(X_main, y_main, X_extra, y_extra, **params)
    c = fold_experiment(X_dirty, y_dirty, **params)
    d = fold_experiment(X_dirty, y_dirty, X_extra, y_extra, **params)
    contam.append({
        "model": name,
        "num_leaves": params["num_leaves"],
        "min_child": params["min_child_samples"],
        "clean_gain": b - a,
        "dirty_gain": d - c,
        "inflation": (d - c) - (b - a),
    })
    print(f"{name:<10} leaves {params['num_leaves']:>3}  "
          f"clean gain {b - a:+.6f}   dirty gain {d - c:+.6f}   "
          f"inflation {(d - c) - (b - a):+.6f}   [{time.time() - T0:.0f}s]")

contam_df = pd.DataFrame(contam)
print("\n" + contam_df.to_string(
    index=False, formatters={k: "{:+.6f}".format
                             for k in ("clean_gain", "dirty_gain", "inflation")}))
print(f"\nfold SD on this file, for scale: {FOLD_SD_MEAN:.6f}")


### Reading the table

Three columns, and they answer three different questions.

`clean_gain` is what appending the extra rows is worth when the two files share no rows:
the honest value of a second synthetic draw. Compare it against the fold SD printed
underneath before calling it real.

`dirty_gain` is what the same append appears to be worth when a small slice of the main
set has exact twins in the appended block. Nothing about that run looks wrong. It does not
warn you. It just scores.

`inflation` is the difference between them — the part of the apparent gain that is bought
by contamination rather than by information. The cell below says which way it went and how
it moved across the capacity rungs.


In [10]:
fold_sd = FOLD_SD_MEAN
for r in contam:
    direction = ("bought" if r["inflation"] > 0 else "cost")
    print(f"{r['model']:<10} (num_leaves={r['num_leaves']}, "
          f"min_child_samples={r['min_child']}): contamination {direction} "
          f"{abs(r['inflation']):.6f} AUC "
          f"= {abs(r['inflation']) / fold_sd:.2f} fold SD")

lo, hi = contam[0], contam[-1]
print(f"\ninflation across the rungs: {lo['inflation']:+.6f} ({lo['model']}) "
      f"-> {hi['inflation']:+.6f} ({hi['model']}), "
      f"a factor of {abs(hi['inflation']) / max(abs(lo['inflation']), 1e-9):,.0f}")

gains = [r["clean_gain"] for r in contam]
print(f"honest gain across the same rungs: {min(gains):+.6f} -> {max(gains):+.6f}, "
      f"spread {max(gains) - min(gains):.6f}")
print(f"honest gains below the fold SD: "
      f"{sum(abs(g) < fold_sd for g in gains)} of {len(gains)}")
print(f"inflations below the fold SD:    "
      f"{sum(abs(r['inflation']) < fold_sd for r in contam)} of {len(contam)}")


So the warning has a shape. "De-duplicate before you append" is not equally
important for everyone: the numbers above are one file's answer to how far it scales with
the capacity a model has to memorise individual rows. Read the three inflation figures
against the fold SD printed with them. If they climb, the people most likely to be
flattered by skipping the de-duplication step are the people fitting the biggest models —
and the honest gain column is the thing to check has *not* climbed with them.

The fix is the same either way. Compare the source file against the competition rows on
the shared feature columns, drop the exact matches, then append what is left to the
training part of each fold.

⚠️ **What this notebook does not know.** Everything above is measured inside the source
file, with contamination I injected on purpose. How many rows S6E9's `train.csv` actually
shares with this file is a different measurement, and it needs the competition data. When
I run it, it goes in its own notebook with its own number, not as an edit to this one.


## `original_clean.csv`

A faithful cleaned copy, not a feature set. Yes/No columns become 0/1 integers, the
target becomes 0/1, numerics become float64, categoricals stay as strings so you can
encode them however you like. No derived columns — this file should be boring enough that
you can mount it without reading the code that made it.


In [11]:
clean = raw.copy()
yes_no = [c for c in clean.columns
          if set(clean[c].astype(str).str.strip().unique()) <= {"Yes", "No", "nan"}]
mapping = {}
for c in yes_no:
    clean[c] = (clean[c].astype(str).str.strip()
                .map({"Yes": 1, "No": 0}).astype("Int8"))
    mapping[c] = "Yes->1, No->0"
for c in num_cols:
    v = pd.to_numeric(clean[c], errors="coerce")
    # keep whole-number columns whole; only the genuinely continuous ones go float
    if v.notna().all() and (v == v.round()).all():
        clean[c] = v.astype("Int64")
    else:
        clean[c] = v.astype("float64")
for c in cat_cols:
    if c not in yes_no:
        clean[c] = clean[c].astype(str)

out_path = os.path.join(WORK, "original_clean.csv")
clean.to_csv(out_path, index=False)

print("recoded:")
for c, how in mapping.items():
    print(f"  {c:<28} {how}")
print(f"\nwrote {out_path}  ({os.path.getsize(out_path) / 1e6:.2f} MB, "
      f"{len(clean):,} rows)")
print("\ndtypes as written:")
print(clean.dtypes.to_string())

check = pd.read_csv(out_path)
assert len(check) == len(raw), "row count changed on the round trip"
assert check[CFG["TARGET"]].isin([0, 1]).all(), "target is not 0/1"
assert abs(check[CFG["TARGET"]].mean() - y.mean()) < 1e-12, "positive rate moved"
print(f"\nround trip checks passed: {len(check):,} rows, "
      f"positive rate {check[CFG['TARGET']].mean():.6f}")


## What I would do with this next

1. Re-run the capacity ladder on `train.csv`. The answer will not be the same, and the
   direction of the change tells you how much of this competition is structure and how
   much is sample size.
2. Compare the competition's distributions against the table above, column by column.
   Where the generator drifted is where this file stops being useful as extra data.
3. Only then decide whether to append. The measurement is cheap and the alternative is
   copying someone else's assumption.

If you fork this, the two lines worth changing first are `LADDER` and `EXTRA_FRAC`.


In [12]:
print(f"total runtime {time.time() - T0:.1f}s")
print(f"best out-of-fold AUC on the source file: {BEST_AUC:.6f}  ({BEST_NAME})")
print(f"mean fold SD:                            {FOLD_SD_MEAN:.6f}")
print("\nappending a second synthetic draw, by model capacity:")
for r in contam:
    print(f"  {r['model']:<10} honest gain {r['clean_gain']:+.6f}   "
          f"contamination inflation {r['inflation']:+.6f}")
print(f"\nwrote {out_path}")
